In [1]:
import pandas as pd
import unicodedata
import re
import lxml


To analyze historical NBA draft performance, I will collect draft data from Basketball Reference covering the 2010–2025 NBA Drafts.

Pandas' `read_html()` function will allow us to extracts HTML tables directly from web pages and converts them into DataFrames.

In [2]:
# Collect draft data from Basketball Reference
all_drafts = []

for year in range(2010, 2026):
    url = f"https://www.basketball-reference.com/draft/NBA_{year}.html"
    df = pd.read_html(url)[0]

    # Flatten MultiIndex columns
    df.columns = df.columns.get_level_values(1)

    df["DraftYear"] = year
    all_drafts.append(df)

drafts = pd.concat(all_drafts, ignore_index=True)

# Remove repeated header rows and round divider rows
drafts = drafts[pd.to_numeric(drafts["Pk"], errors="coerce").notna()].copy()
drafts["Pk"] = drafts["Pk"].astype(int)

drafts.to_csv("nba_draft_data.csv", index=False)

To prepare for merging, player names from the combine dataset are converted from "Last, First" format to "First Last" format so they match the naming convention used in the draft dataset. The same is done for draft year.

I also renamed the `Year` column so it aligns with the `DraftYear` column in the draft data to prepare for merging.

In [3]:
# Load and clean combine data
combine = pd.read_csv("data/NBA_Draft_Combine_2000_2024_nl.csv")

def last_first_to_first_last(name):
    parts = str(name).split(", ")

    if len(parts) == 2:
        return f"{parts[1]} {parts[0]}"

    return name

combine["Player"] = combine["Player Name (Last Name, First Name)"].apply(
    last_first_to_first_last
)

combine = combine.rename(columns={"Year": "DraftYear"})

combine = combine[
    combine["DraftYear"].between(2010, 2025)
].copy()

combine.to_csv("data/combine_data.csv", index=False)

Here we create a function to standardize names by remvoing accents, dropping suffixes, removing punctuation, collapsing whitespaces, and applying lowercase to all letters to help ensure no matches will be missed.

In [4]:
# Better name normalization for matching
def normalize_name(name):
    name = str(name)

    # Remove accents: Vásquez -> Vasquez
    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("ascii")

    name = name.lower()
 
    # Remove suffixes: Jr., III, IV, etc.
    name = re.sub(r"\b(jr|sr|ii|iii|iv)\.?\b", "", name)

    # Remove punctuation and weird characters
    name = re.sub(r"[^a-z ]", " ", name)

    # Collapse extra spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

drafts["Player_clean"] = drafts["Player"].apply(normalize_name)
combine["Player_clean"] = combine["Player"].apply(normalize_name)

Merge our draft and combine datasets on the normalized `Player_clean` columnn and `DraftYear`.

We check how much matched using the `Position` column which only exists in the combine dataset (merge is a left join on draft dataset).

Note: Unmatched rows mostly represent players who did not appear in the combine dataset, including international players, players who skipped the combine, or records with alternate names/nicknames.

In [5]:
# Merge using cleaned player name + draft year
master = drafts.merge(
    combine,
    on=["Player_clean", "DraftYear"],
    how="left",
    suffixes=("_draft", "_combine")
)

# Position would be null if no match exists
matched = master["Position"].notna().sum()

print(f"Matched: {matched}")
print(f"Total draft picks: {len(master)}")
print(f"Match Rate: {matched / len(master):.2%}")

Matched: 661
Total draft picks: 956
Match Rate: 69.14%


This section downloads player-level college basketball statistics from the public toRvik dataset for seasons 2014–2023. Three categories of statistics are collected:

- <u>Advanced metrics</u>

This includes efficiency and impact metrics. It tells us how much a player helped their team on offense and defense, adjusted for pace and context. Some columns include `stops` for defensive stops, `bpm` for box plus-minus, and `ortg` for offensive rating.

        

- <u>Traditional box score statistics</u>

Classic per-game counting stats and what you’d see on a game box score or player page. 
Some columns include `mpg` for minutes per game, `tov` for turnovers, and `fg_pct` for field goal percentage.

- <u>Shooting statistics</u>

Statistics on where and how a player scores, not just how much. Some columns include `usg` for usage rate, `efg` foreffective FG%, and `dunk_m/a` for dunk frquency and success.


Note: Any two player can have the same pgg, but that is not the whole story. Both can have very different profiles.


In [6]:
def load_torvik_player_season(year, stat):
    url = f"https://raw.githubusercontent.com/andreweatherman/toRvik-data/main/player_season/{year}/{stat}_{year}.parquet"
    df = pd.read_parquet(url)
    df["year"] = year
    return df

years = range(2014, 2024)  # 2014 through 2023

advanced = pd.concat(
    [load_torvik_player_season(year, "advanced") for year in years],
    ignore_index=True
)

box = pd.concat(
    [load_torvik_player_season(year, "box") for year in years],
    ignore_index=True
)

shooting = pd.concat(
    [load_torvik_player_season(year, "shooting") for year in years],
    ignore_index=True
)

advanced.to_csv("college_advanced_2014_2023.csv", index=False)
box.to_csv("college_box_2014_2023.csv", index=False)
shooting.to_csv("college_shooting_2014_2023.csv", index=False)

C:\Users\johnn\AppData\Local\Temp\ipykernel_25300\1279704677.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  advanced = pd.concat(


Match rows using the same player id and the same college year. 

The same player could appear in multiple seasons, so you don't want to merge only on id. We want the right player and the right season.

In [7]:
# Explore potential columns to merge on
set(advanced.columns) & set(box.columns) & set(shooting.columns)

merge_keys = ["id", "year"]

# Keep every row from advanced, even if there is no match in box or shooting. The suffixes part handles duplicate column names.
college = (
    advanced
    .merge(box, on=merge_keys, how="left", suffixes=("", "_box"))
    .merge(shooting, on=merge_keys, how="left", suffixes=("", "_shooting"))
)

college.to_csv("college_stats_2014_2023.csv", index=False)

To connect college performance data with NBA draft outcomes, drafted players were matched to their corresponding college statistics using standardized player names.

Since many players appear in the college dataset multiple times (one row per season), a matching process was used to identify the most relevant college season for each draft prospect. Candidate matches were restricted to seasons occurring before or during the player's draft year.


In [ ]:
college["Player_clean"] = college["player"].apply(normalize_name)

# Give every draft/combine row a stable ID before merging college stats
master = master.reset_index(drop=True).copy()
master["draft_row_id"] = master.index

# We are only provided 2014-2023 data from toRvik. 
draft_for_college = master[
    master["DraftYear"].between(2014, 2023)
    & master["Player_draft"].notna()
].copy()

# Match each drafted player to all possible college seasons
college_candidates = draft_for_college[
    ["draft_row_id", "DraftYear", "Pk", "Player_clean"]
].merge(
    college,
    on="Player_clean",
    how="left"
)

# Keep only seasons before or during the draft year
college_candidates = college_candidates[
    college_candidates["year"].notna()
    & (college_candidates["year"] <= college_candidates["DraftYear"])
].copy()

#  prefer the season closest to draft year (usually final college year)
college_candidates["year_gap"] = (
    college_candidates["DraftYear"] - college_candidates["year"]
)

# Helpful tie-breaker when names repeat
college_candidates["college_pick_num"] = pd.to_numeric(
    college_candidates["pick"], errors="coerce"
)

college_candidates["draft_pick_num"] = pd.to_numeric(
    college_candidates["Pk"], errors="coerce"
)

# prefer rows where mock draft pick matches actual pick
college_candidates["pick_match"] = (
    college_candidates["college_pick_num"].round().astype("Int64")
    .eq(college_candidates["draft_pick_num"].astype("Int64"))
    .fillna(False)
    .astype(bool)
)

Candidate seasons were prioritized using the following criteria:

1. Whether the college record matched the player's draft pick number.
2. How close the college season was to the player's draft year.
3. Total minutes played during the season.
4. Total games played during the season.

After ranking all possible matches, only the highest-priority record for each drafted player was retained. This approach links every prospect to the college season most representative of their pre-draft performance while preventing duplicate player records in the final dataset.

In [14]:
#prefer bigger-minute season if tied, prefer more games played if still tied
college_candidates = college_candidates.sort_values(
    ["draft_row_id", "pick_match", "year_gap", "min", "g"],
    ascending=[True, False, True, False, False]
)

# keep only the top-ranked row per draf pick
college_best = college_candidates.drop_duplicates("draft_row_id").copy()

After selecting one college season per drafted player, I kept the most relevant college performance columns, renamed them with a `college_` prefix, and left-merged them into the master dataset using `draft_row_id`. This preserves every drafted player in the dataset while adding college statistics when available. Players without matched college data, such as international players or players from non-college pathways, remain in the dataset with missing college values.

In [18]:
college_cols = [
    "draft_row_id",
    "year",
    "year_gap",
    "player",
    "team",
    "conf",
    "pos",
    "exp",
    "g",
    "min",
    "porpag",
    "dporpag",
    "ortg",
    "drtg",
    "obpm",
    "dbpm",
    "bpm",
    "oreb",
    "dreb",
    "ast",
    "to",
    "blk",
    "stl",
    "ftr",
    "pfr",
    "mpg",
    "ppg",
    "fg_pct",
    "rpg",
    "apg",
    "tov",
    "ast_to",
    "spg",
    "bpg",
    "usg",
    "efg",
    "ts",
    "ft_pct",
    "two_pct",
    "three_pct",
    "rim_pct",
    "mid_pct",
]

# pull out the columns we want
college_features = college_best[college_cols].copy()

# rename columns so they don't clash with draft/nba columns
college_features = college_features.rename(
    columns={
        col: f"college_{col}"
        for col in college_features.columns
        if col != "draft_row_id"
    }
)

# Delete any old college stats that might already be there (if re-running)
old_college_cols = [
    col for col in master.columns
    if col.startswith("college_") or col == "has_college_stats"
]

master = master.drop(columns=old_college_cols, errors="ignore")

master = master.merge(
    college_features,
    on="draft_row_id",
    how="left"
)

# flag who got college data
master["has_college_stats"] = master["college_player"].notna()

After merging draft, combine, and college statistics into the master dataset, I created a final Tableau-ready CSV. Blank draft rows from Basketball Reference were removed so each row represents one real drafted player. I also added indicator columns for whether each player has combine data or college statistics, which will make it easier to filter and interpret missing data in Tableau.

Several columns were renamed to make the dataset easier to use in visualizations. For example, NBA per-game statistics were renamed as `nba_ppg`, `nba_rpg`, `nba_apg`, and `nba_mpg`, while combine fields were renamed with clearer labels. The final dataset is exported as `tableau_nba_draft_master.csv`, which will be used for Tableau dashboards and player comparisons.

In [19]:
tableau_data = master.copy()

# Remove blank Basketball Reference draft rows
tableau_data = tableau_data[tableau_data["Player_draft"].notna()].copy()

# Helpful flags for Tableau filters
tableau_data["has_combine_data"] = tableau_data["uniqueID"].notna()
tableau_data["has_college_stats"] = tableau_data["college_player"].notna()

# Optional cleaner player column
tableau_data = tableau_data.rename(columns={
    "Player_draft": "player",
    "Player_combine": "combine_player",
    "Position": "combine_position",
    "Height in Ft & In": "combine_height",
    "Weight in lbs": "combine_weight_lbs",
    "PTS.1": "nba_ppg",
    "TRB.1": "nba_rpg",
    "AST.1": "nba_apg",
    "MP.1": "nba_mpg",
    "FG%": "nba_fg_pct",
    "3P%": "nba_3p_pct",
    "FT%": "nba_ft_pct",
})

print(tableau_data.shape)
print(tableau_data["has_combine_data"].mean())
print(tableau_data["has_college_stats"].mean())

tableau_data.to_csv("tableau_nba_draft_master.csv", index=False)

(953, 80)
0.6935991605456453
0.48478488982161594


In [22]:
df = tableau_data

# Coverage - A look into how complete our data is
print(f"Total players: {len(df)}")
print(f"Draft years: {df['DraftYear'].min()} – {df['DraftYear'].max()}")
print(f"Has combine data: {df['has_combine_data'].mean():.1%}")
print(f"Has college stats: {df['has_college_stats'].mean():.1%}")
print(f"Has both: {(df['has_combine_data'] & df['has_college_stats']).mean():.1%}")

Total players: 953
Draft years: 2010 – 2025
Has combine data: 69.4%
Has college stats: 48.5%
Has both: 37.0%


In [27]:
# Among all drafted players in our dataset who played at least 50 NBA games, who are the 10 best by career BPM?

df["G"] = pd.to_numeric(df["G"], errors="coerce")
df["BPM"] = pd.to_numeric(df["BPM"], errors="coerce")

qualified = df[df["G"].fillna(0) >= 50]

qualified.nlargest(10, "BPM")[
    ["player", "DraftYear", "Pk", "College", "G", "BPM", "WS/48", "has_college_stats"]
]

,player,DraftYear,Pk,College,G,BPM,WS/48,has_college_stats
280,Nikola Jokić,2014,41,NaN,810.0,10.6,.265,False
482,Luka Dončić,2018,3,NaN,514.0,7.8,.178,False
778,Victor Wembanyama,2023,1,NaN,181.0,7.4,.152,False
194,Giannis Antetokounmpo,2013,15,NaN,895.0,6.8,.206,False
242,Joel Embiid,2014,3,Kansas,490.0,6.7,.212,True
74,Kawhi Leonard,2011,15,San Diego State,798.0,6.6,.212,False
490,Shai Gilgeous-Alexander,2018,11,Kentucky,530.0,6.3,.207,True
120,Anthony Davis,2012,1,Kentucky,807.0,5.8,.208,False
89,Jimmy Butler,2011,30,Marquette,907.0,5.0,.207,False
611,Tyrese Haliburton,2020,12,Iowa State,333.0,4.8,.164,True
